<a href="https://colab.research.google.com/github/ankitaguptacs24-ai/AI_LAB/blob/main/IBF24CS057_Week5_PropositionalLogic_TruthTableEnumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import re

# ==========================================
# 1. EXPRESSION EVALUATOR WITH OPERATOR OVERLOADING
# ==========================================
class BoolVar:
    """
    Overloads standard Python bitwise and comparison operators
    to represent formal propositional logic evaluation safely.
    """
    def __init__(self, val):
        self.val = bool(val)
    def __and__(self, other):
        return BoolVar(self.val and other.val)
    def __or__(self, other):
        return BoolVar(self.val or other.val)
    def __invert__(self):
        return BoolVar(not self.val)
    def __rshift__(self, other):
        # Implication: P >> Q is equivalent to ~P | Q
        return BoolVar(not self.val or other.val)
    def __eq__(self, other):
        # Biconditional (IFF): P == Q
        return BoolVar(self.val == other.val)
    def __bool__(self):
        return self.val

def evaluate(expr, model):
    """Evaluates a logic sentence string using the provided variable mapping model."""
    env = {sym: BoolVar(val) for sym, val in model.items()}
    try:
        return bool(eval(expr, {}, env))
    except Exception as e:
        print(f"Error evaluating '{expr}': {e}")
        return False

# ==========================================
# 2. TRUTH TABLE ENUMERATION ENTAILMENT ALGORITHM
# ==========================================
def get_symbols(expressions):
    """Extracts unique propositional variables from logic sentences."""
    symbols = set()
    for expr in expressions:
        words = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', expr)
        for word in words:
            if word not in ['and', 'or', 'not', 'True', 'False']:
                symbols.add(word)
    return sorted(list(symbols))

def tt_entails(kb, q):
    """Returns True if KB entails q, and False otherwise."""
    symbols = get_symbols(kb + [q])
    return tt_check_all(kb, q, symbols, {})

def tt_check_all(kb, q, symbols, model):
    """Recursive helper to evaluate all possible truth value permutations."""
    if not symbols:
        # Base case: All variables assigned values
        if all(evaluate(sentence, model) for sentence in kb):
            return evaluate(q, model)
        return True
    else:
        P = symbols[0]
        rest = symbols[1:]
        # Branch into True and False assignments for symbol P
        return (tt_check_all(kb, q, rest, {**model, P: True}) and
                tt_check_all(kb, q, rest, {**model, P: False}))

# ==========================================
# 3. VERIFICATION AND TESTING
# ==========================================
print("--- Test Case: Valid Entailment (Modus Ponens) ---")
kb = ["P >> Q", "Q >> R", "P"]
query = "R"
print(f"KB: {kb}")
print(f"Query: {query}")
print(f"Does KB entail Query? {tt_entails(kb, query)}")

--- Test Case: Valid Entailment (Modus Ponens) ---
KB: ['P >> Q', 'Q >> R', 'P']
Query: R
Does KB entail Query? True


In [12]:
# ==========================================
# 4. TRUTH TABLE GENERATION AND VISUALIZATION
# ==========================================

def print_truth_table(kb, q):
    symbols = get_symbols(kb + [q])

    # Header
    header = list(symbols) + kb + ["KB (AND)", q, "KB |= Query?"]
    print(" | ".join(f"{h:^10}" for h in header))
    print("-" * (13 * len(header)))

    # Generate all truth combinations (2^n models)
    n = len(symbols)
    all_models = []
    for i in range(1 << n):
        model = {}
        for j, sym in enumerate(symbols):
            # Assign True or False based on binary representation bits
            model[sym] = bool((i >> (n - 1 - j)) & 1)
        all_models.append(model)

    for model in all_models:
        # Evaluate each sentence in KB
        kb_vals = [evaluate(sentence, model) for sentence in kb]
        # Evaluate query
        q_val = evaluate(q, model)
        # Combined KB value
        kb_combined = all(kb_vals)

        # Check if row satisfies entailment requirement
        # Entailment is violated ONLY if KB is True but Query is False
        row_status = "OK" if (not kb_combined or q_val) else "VIOLATED"
        if kb_combined and q_val:
            row_status = "ENTAILED (True/True)"

        # Prepare row values for printing
        row_vals = [str(model[sym]) for sym in symbols]
        row_vals += [str(val) for val in kb_vals]
        row_vals += [str(kb_combined), str(q_val), row_status]

        print(" | ".join(f"{val:^10}" for val in row_vals))

# Display the truth table for the default test case
print_truth_table(kb, query)

    P      |     Q      |     R      |   P >> Q   |   Q >> R   |     P      |  KB (AND)  |     R      | KB |= Query?
---------------------------------------------------------------------------------------------------------------------
  False    |   False    |   False    |    True    |    True    |   False    |   False    |   False    |     OK    
  False    |   False    |    True    |    True    |    True    |   False    |   False    |    True    |     OK    
  False    |    True    |   False    |    True    |   False    |   False    |   False    |   False    |     OK    
  False    |    True    |    True    |    True    |    True    |   False    |   False    |    True    |     OK    
   True    |   False    |   False    |   False    |    True    |    True    |   False    |   False    |     OK    
   True    |   False    |    True    |   False    |    True    |    True    |   False    |    True    |     OK    
   True    |    True    |   False    |    True    |   False    |    True   